In [1]:
import json
import re

# GPT output simulation
gpt_outputs = {
    'L5G1_L4G1_L4G3': 
    '- Group L5G1 Common Features: <<retinal edges with pale hues and faint, branching vessels>>\n- Group L4G1 Common Features: <<pale blue-green coloration with soft gradients>>\n- Group L4G3 Common Features: <<faint, curved lines resembling curved vascular structures>>\n\n- Feature Evolution:\n    - Group L4G1: <<There is no match>>\n    - Group L4G3: <<curved vascular structures evolve into branching vessels>>\n\n- Caption: <<Group L5G1 combines pale hues with branching vessels, integrating features of curved vascular structures.>>',
    'L4G1_L3G0_L3G1': 
    '- Group L4G1 Common Features: <<Prominent blood vessels, optic disc.>>\n- Group L3G0 Common Features: <<Curved blood vessels, orange background.>>\n- Group L3G1 Common Features: <<Diffuse orange background, subtle vascular patterns.>>\n\n- Feature Evolution:\n    - Group L3G0: <<Prominent blood vessels from curved blood vessels on an orange background.>>\n    - Group L3G1: <<Optic disc evolution from diffuse orange background.>>\n\n- Caption: <<The common features in Group L4G1 involve prominent blood vessels and the optic disc, evolved from curved vascular patterns and an orange retinal background.>>',
    'L4G3_L3G0_L3G2': 
    '- Group L4G3 Common Features: <<Curved lines against light background, slight gradient colors>>\n- Group L3G0 Common Features: <<Curved red lines, pale bluish background>>\n- Group L3G2 Common Features: <<Curved orange lines, reddish-orange background>>\n\n- Feature Evolution:\n    - Group L3G0: <<Curved lines against light background>>\n    - Group L3G2: <<Curved lines with gradient background>>\n\n- Caption: <<Group L4G3 shows curved lines with a slight color gradient, evolving from the distinct lines and background colors of Groups L3G0 and L3G2.>>'
}

In [2]:
import re
import json

group_hierarchy = {}
group_metadata = {}
group_captions = {}
feature_evolution = {}
feature_weights = {} 

# parse LLM output
for key, text in gpt_outputs.items():
    groups = key.split("_")
    parent = groups[0]
    children = groups[1:]

    # initial hierarchy
    group_hierarchy.setdefault(parent, []).extend(children)

    # parse Common Features
    feature_matches = re.findall(r"- Group (\w+) Common Features: <<(.+?)>>", text)
    for group, description in feature_matches:
        group_metadata[group] = {
            "description": description.strip(),
            "images": []
        }

    # parse Feature Evolution
    evolution_matches = re.findall(r"- Group (\w+): <<(.+?)>>", text.split("- Feature Evolution:")[-1])
    for child_group, relation in evolution_matches:
        feature_evolution.setdefault(parent, {})[child_group] = relation.strip()
        feature_weights.setdefault(parent, {})[child_group] = 0.5

    # parse Caption, key is full group key（parent + all children）
    caption_match = re.search(r"- Caption: <<(.+?)>>", text)
    if caption_match:
        full_group_key = '_'.join([parent] + children)
        group_captions[full_group_key] = caption_match.group(1).strip()

# save
with open("group_hierarchy.json", "w", encoding="utf-8") as f:
    json.dump(group_hierarchy, f, indent=4, ensure_ascii=False)

with open("group_metadata.json", "w", encoding="utf-8") as f:
    json.dump(group_metadata, f, indent=4, ensure_ascii=False)

with open("group_captions.json", "w", encoding="utf-8") as f:
    json.dump(group_captions, f, indent=4, ensure_ascii=False)

with open("feature_evolution.json", "w", encoding="utf-8") as f:
    json.dump(feature_evolution, f, indent=4, ensure_ascii=False)

with open("feature_weights.json", "w", encoding="utf-8") as f:
    json.dump(feature_weights, f, indent=4, ensure_ascii=False)

# print for verification
print("[Hierarchy]")
print(json.dumps(group_hierarchy, indent=2, ensure_ascii=False))
print("\n[Metadata]")
print(json.dumps(group_metadata, indent=2, ensure_ascii=False))
print("\n[Captions]")
print(json.dumps(group_captions, indent=2, ensure_ascii=False))
print("\n[Feature Evolution]")
print(json.dumps(feature_evolution, indent=2, ensure_ascii=False))
print("\n[Feature Weights]")
print(json.dumps(feature_weights, indent=2, ensure_ascii=False))


[Hierarchy]
{
  "L5G1": [
    "L4G1",
    "L4G3"
  ],
  "L4G1": [
    "L3G0",
    "L3G1"
  ],
  "L4G3": [
    "L3G0",
    "L3G2"
  ]
}

[Metadata]
{
  "L5G1": {
    "description": "retinal edges with pale hues and faint, branching vessels",
    "images": []
  },
  "L4G1": {
    "description": "Prominent blood vessels, optic disc.",
    "images": []
  },
  "L4G3": {
    "description": "Curved lines against light background, slight gradient colors",
    "images": []
  },
  "L3G0": {
    "description": "Curved red lines, pale bluish background",
    "images": []
  },
  "L3G1": {
    "description": "Diffuse orange background, subtle vascular patterns.",
    "images": []
  },
  "L3G2": {
    "description": "Curved orange lines, reddish-orange background",
    "images": []
  }
}

[Captions]
{
  "L5G1_L4G1_L4G3": "Group L5G1 combines pale hues with branching vessels, integrating features of curved vascular structures.",
  "L4G1_L3G0_L3G1": "The common features in Group L4G1 involve prominent 